# 🛡️ SentinelSpam: SMS Spam Detection & NLP Machine Learning Pipeline

## 📌 Project Overview
This notebook demonstrates an end-to-end Machine Learning pipeline to classify SMS messages into **Ham** (Legitimate) or **Spam**.

### Pipeline Steps:
1. **Data Cleaning & Normalization**
2. **Exploratory Data Analysis (EDA)**
3. **Text Preprocessing & Feature Engineering (TF-IDF)**
4. **Classifier Benchmarking** (Naive Bayes, SVM, Random Forest, Logistic Regression, Extra Trees)
5. **Model Evaluation & Export**

In [ ]:
# Import Required Packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Preprocessing helper
from src.preprocessing import clean_text, extract_metadata_features

%matplotlib inline
sns.set_style('whitegrid')

## 1. Load & Inspect Dataset

In [ ]:
# Load SMS Dataset
df = pd.read_csv('../data/sms_spam_dataset.csv')
print(f"Dataset Shape: {df.shape}")
df.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Encode Labels
df['target'] = df['label'].map({'ham': 0, 'spam': 1})

# Class Distribution
plt.figure(figsize=(6, 4))
sns.countplot(x='label', data=df, palette=['#2E7D32', '#FF4D4D'])
plt.title('Distribution of Ham vs Spam Messages')
plt.show()

print(df['label'].value_counts(normalize=True) * 100)

In [ ]:
# Message Length Feature Analysis
df['num_characters'] = df['text'].apply(len)
df['num_words'] = df['text'].apply(lambda x: len(x.split()))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df[df['target'] == 0]['num_characters'], ax=ax[0], color='green', label='Ham', kde=True)
sns.histplot(df[df['target'] == 1]['num_characters'], ax=ax[0], color='red', label='Spam', kde=True)
ax[0].set_title('Character Count Distribution')
ax[0].legend()

sns.histplot(df[df['target'] == 0]['num_words'], ax=ax[1], color='green', label='Ham', kde=True)
sns.histplot(df[df['target'] == 1]['num_words'], ax=ax[1], color='red', label='Spam', kde=True)
ax[1].set_title('Word Count Distribution')
ax[1].legend()
plt.show()

## 3. Text Preprocessing & TF-IDF Extraction

In [ ]:
print("Cleaning text messages...")
df['cleaned_text'] = df['text'].apply(clean_text)

# TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1, 2))
X = tfidf.fit_transform(df['cleaned_text']).toarray()
y = df['target'].values

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

## 4. Model Training & Classifier Evaluation

In [ ]:
models = {
    'Multinomial Naive Bayes': MultinomialNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Extra Trees': ExtraTreesClassifier(n_estimators=100, random_state=42),
    'SVC (Sigmoid)': SVC(kernel='sigmoid', gamma=1.0, probability=True, random_state=42)
}

results = []
for name, clf in models.items():
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds, zero_division=0),
        'Recall': recall_score(y_test, preds, zero_division=0),
        'F1-Score': f1_score(y_test, preds, zero_division=0)
    })

results_df = pd.DataFrame(results).sort_values(by=['Precision', 'Accuracy'], ascending=False)
results_df

## 5. Model Serialization & Export

In [ ]:
# Export artifacts to models/
os.makedirs('../models', exist_ok=True)
best_clf = MultinomialNB()
best_clf.fit(X_train, y_train)

joblib.dump(tfidf, '../models/tfidf_vectorizer.pkl')
joblib.dump(best_clf, '../models/spam_classifier.pkl')
print("Model and vectorizer saved successfully!")